In [12]:
import Data_Loader
from importlib import reload
reload(Data_Loader)
import os
import random

folder = "/mnt/c/Users/uhewm/Desktop/ProjectHGT/simulation_chunks"
#folder = "/mnt/c/ProjectHGT/simulation_chunks"

files = [entry.path for entry in os.scandir(folder) if entry.is_file()]

if len(files) > 5000:
    files = random.sample(files, 5000)

data_graphs = []
for f in files:
    try:
        d = Data_Loader.load_file(f)
        Data_Loader.aggregate_sequences(d)
        data_graphs.append(d)
    except Exception as e:
        print(f"Fehler beim Laden von {f}: {e}")

data = []
for G in data_graphs:
    try:
        dat = Data_Loader.RecipientFinder_graph_to_dataset(G)
        data.append(dat)
    except Exception as e:
        print(f"Fehler beim Laden von {G}: {e}")


"""
for node, attrs in data_graphs[0].nodes(data=True):
    seq = attrs.get("sequences", None)
    if seq is not None:
        print(node, seq[:50])   # nur die ersten 50 Stellen
"""

data_sample = random.choice(data)
#example = load_file(file)


In [78]:
data_graphs[0].

In [13]:
import torch
from torch import nn

class BinaryTreeLSTMCell(nn.Module):
    """
    Binary Tree-LSTM cell (Tai et al., 2015).
    """

    def __init__(self, hidden_dim, internal_node_data_dim):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.internal_node_data_dim = internal_node_data_dim
        
        def lin():
            return nn.Linear(hidden_dim + internal_node_data_dim, hidden_dim)

        self.i = lin()
        self.f = lin()
        self.u = lin()
        self.o = lin()

    def forward(self, x, hl, cl, hr, cr):
        """
        hl, cl: left child hidden & cell
        hr, cr: right child hidden & cell

        The left and right order does not matter (invariant).
        """

        hsum = hl + hr

        i = torch.sigmoid(self.i(torch.cat((x, hsum), dim=1)))
        fl = torch.sigmoid(self.f(torch.cat((x, hl), dim=1)))
        fr = torch.sigmoid(self.f(torch.cat((x, hr), dim=1)))
        o = torch.sigmoid(self.o(torch.cat((x, hsum), dim=1)))
        u = torch.tanh(self.u(torch.cat((x, hsum), dim=1)))

        c = i * u + fl * cl + fr * cr

        h = o * torch.tanh(c)

        return h, c


In [14]:
import torch
from torch import nn
import torch.nn.functional as F
from torch_geometric.nn import MessagePassing
import numpy as np
from collections import defaultdict
reload(Data_Loader)

class RecipientFinder(nn.Module):
    """
    A model that stacks several ParentChildFusionLayers followed by
    fully connected layers for prediction.

    Parameters
    ----------
    in_channels : int
        Dimensionality of the input node features.
    fc_hidden_channels : int
        Dimensionality after the ParentChildFusionLayers.
    num_fusion_layers : int
        Number of ParentChildFusionLayers to apply sequentially.
    dropout : float
        Dropout probability for the fully connected classifier.
    """

    def __init__(self, internal_node_data_dim, 
                 conv_out_channels = 4, conv_kernel_size = 5, fc_hidden_channels = 16, 
                 num_fc_layers = 3, dropout=0.1, max_number_of_snps = 300, len_alphabet = 5):

        super().__init__()

        self.max_number_of_snps = max_number_of_snps
        self.len_alphabet = len_alphabet
        self.internal_node_data_dim = internal_node_data_dim
        
        # ----- CNN Layer -----

        self.conv = nn.Conv2d(
            in_channels=1,
            out_channels=conv_out_channels,
            kernel_size=(len_alphabet, conv_kernel_size),
            stride=(1, conv_kernel_size),
            padding=0
        )

        dummy_x = torch.zeros(1, 1, len_alphabet, max_number_of_snps)
        with torch.no_grad():
            dummy_out = self.conv(dummy_x)
        dim_after_conv = dummy_out.shape
        current_dim = dim_after_conv[1] * dim_after_conv[3]
        
        # ----- LSTM Cell: Bottom-up traversal -----

        self.conv_lstm = nn.Conv2d(
            in_channels=1,
            out_channels=conv_out_channels,
            kernel_size=(len_alphabet, conv_kernel_size),
            stride=(1, conv_kernel_size),
            padding=0
        )
        
        self.Tree_LSTM = BinaryTreeLSTMCell(hidden_dim = current_dim, internal_node_data_dim = internal_node_data_dim)
        current_dim = 2* current_dim + internal_node_data_dim
        #current_dim = current_dim + internal_node_data_dim

        # ----- GCN Layer -----

        self.fusion_layer = Data_Loader.ParentChildFusionLayer_RecipientFinder(in_dim=current_dim)
        current_dim = 3 * current_dim

        # ----- Fully Connected Layers -----
        self.fc_layers = nn.ModuleList()
        self.dropout = dropout

        # First FC layer: (current_dim → fc_hidden_channels) OR directly → 1
        if num_fc_layers == 1:
            self.fc_layers.append(nn.Linear(current_dim, 1))
        else:
            # First hidden layer
            self.fc_layers.append(nn.Linear(current_dim, fc_hidden_channels))
            # Middle hidden FC layers
            for _ in range(num_fc_layers - 2):
                self.fc_layers.append(nn.Linear(fc_hidden_channels, fc_hidden_channels))
            # Final output layer
            self.fc_layers.append(nn.Linear(fc_hidden_channels, 1))

    def forward(self, x, internal_node_data, level, edge_index):
        """
        Apply stacked fusion layers followed by linear classifiers.
        """      
        N = x.size(0)
    
        # --- Restore one-hot structure ---
        x = x.view(N, self.max_number_of_snps, self.len_alphabet)
        x = x.permute(0, 2, 1)
        x = x.unsqueeze(1)   # [N, 1, 5, max_snps]
    
        # ----- CNN -----
        x_internal = self.conv(x)
        x_internal = F.relu(x_internal)
    
        x_internal = x_internal.squeeze(2)     # [N, C, W]
        x_internal = x_internal.flatten(start_dim=1)  # [N, C * W]
        
        # ----- LSTM Cell: Bottom-up traversal -----


        x_lstm = self.conv_lstm(x)
        x_lstm = F.relu(x_lstm)
    
        x_lstm = x_lstm.squeeze(2)     # [N, C, W]
        x_lstm = x_lstm.flatten(start_dim=1)  # [N, C * W]

        children = self.build_children(edge_index, N)
        
        recurrent_data = self.tree_lstm_bottom_up(
            x = x_lstm,
            children = children,
            internal_node_data = internal_node_data,
            level = level,
            cell = self.Tree_LSTM
        )
        
        x = torch.cat((x_internal, recurrent_data, internal_node_data), dim=1)

        #x = torch.cat((x_internal, internal_node_data), dim=1)

        # ----- GCN Layer -----

        x = self.fusion_layer(x, edge_index)
        
        # ----- Fully Connected Layers -----
        for i, layer in enumerate(self.fc_layers):
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = layer(x)
            if i < len(self.fc_layers) - 1:
                x = F.relu(x)

        return x.view(-1)

    def build_children(self, edge_index, num_nodes):
        """
        edge_index: [2, E] tensor, child -> parent
        """
        children = [[] for _ in range(num_nodes)]
    
        src, dst = edge_index
        for c, p in zip(src.tolist(), dst.tolist()):
            children[p].append(c)
    
        return children
    
    def build_levels(self, level):
        levels = defaultdict(list)
        for v, d in enumerate(level):
            levels[int(d)].append(v)
        return levels
        

    def tree_lstm_bottom_up(self, x, children, internal_node_data, level, cell):
        """
        x: [N, D] leaf embeddings (CNN output), internal nodes arbitrary
        children: list of length N, [] or [l, r]
        level: [N] integer level values
        """
        device = x.device
        N, D = x.shape
    
        h = torch.zeros(N, D, device=device)
        c = torch.zeros(N, D, device=device)
    
        levels = self.build_levels(level)
    
        max_level = max(levels.keys())
    
        # level = 0 → leaves
        leaf_nodes = levels[0]
        h[leaf_nodes] = x[leaf_nodes]
    
        # process internal levels
        for d in range(1, max_level + 1):
            nodes = levels[d]
    
            # gather children indices
            left = torch.tensor(
                [children[v][0] for v in nodes],
                device=device
            )
            right = torch.tensor(
                [children[v][1] for v in nodes],
                device=device
            )
            
            node_data = internal_node_data[nodes]
    
            hl = h[left]
            cl = c[left]
            hr = h[right]
            cr = c[right]
            
    
            h_new, c_new = cell(node_data, hl, cl, hr, cr)
    
            h[nodes] = h_new
            c[nodes] = c_new
   
        return h

i = random.choice(range(100))
#test_model = RecipientFinder(internal_node_data_dim = data[i].internal_node_data.shape[1])
test_model = RecipientFinder(num_nodes = data[i].x.shape[0], internal_node_data_dim = data[i].internal_node_data.shape[1])
#print(model)
test_model(data[i].x, data[i].internal_node_data, data[i].level, data[i].edge_index)

tensor([ 0.2019,  0.1758,  0.2016,  0.2076,  0.2058,  0.2092,  0.2057,  0.1452,
         0.1490,  0.2025,  0.2041,  0.2046,  0.2152,  0.2010,  0.2182,  0.1309,
         0.2020,  0.1795,  0.2032,  0.2065,  0.1311,  0.1714,  0.2011,  0.1920,
         0.2024,  0.1663,  0.2067,  0.1908,  0.1792,  0.2066,  0.1605,  0.2118,
         0.2161,  0.1779,  0.2029,  0.1882,  0.1564,  0.2033,  0.1916,  0.2130,
         0.2120,  0.2169,  0.2184,  0.2235,  0.2144,  0.2373,  0.1958,  0.1721,
         0.2173,  0.2409,  0.2129,  0.1956,  0.2134,  0.1888,  0.2096,  0.1778,
         0.1874,  0.2018,  0.1794,  0.1689,  0.1869,  0.1794,  0.2021,  0.2206,
         0.1499,  0.2089,  0.2021,  0.2017,  0.2013,  0.2016,  0.2082,  0.2110,
         0.2014,  0.2114,  0.2021,  0.1854,  0.2052,  0.2179,  0.1920,  0.2048,
         0.1933,  0.1628,  0.2181,  0.2043,  0.2028,  0.2178,  0.1616,  0.1752,
         0.1774,  0.1669,  0.2082,  0.2062,  0.1929,  0.1479,  0.2028,  0.1831,
         0.2072,  0.1661,  0.1697,  0.13

In [19]:
data[0].internal_node_data

tensor([[1.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
        [1.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
        [1.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
        [1.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
        [1.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
        [1.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
        [1.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
        [1.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
        [1.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
        [1.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
        [1.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
        [1.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
        [1.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
        [1.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
        [1.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
        [1.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
        [1.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
        [1.000

In [22]:
import Data_Loader
reload(Data_Loader)
from torch_geometric.loader import DataLoader


# === 1. Modell, Optimizer, Loss ===

#model = RecipientFinder(num_nodes = data[0].x.shape[0], internal_node_data_dim = data[0].internal_node_data.shape[1])
model = RecipientFinder(internal_node_data_dim = data[0].internal_node_data.shape[1])

optimizer = torch.optim.Adam(model.parameters(), lr=3*1e-3)

# Klassengewichte berechnen (gegen Ungleichgewicht)
all_labels = torch.cat([g.y for g in data])
ratio = (len(all_labels) - all_labels.sum()) / all_labels.sum()
pos_weight = torch.tensor((ratio**0.3), dtype=torch.float)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

print(f"Pos Weight: {pos_weight.item():.2f}")

# === 2. Training & Evaluation ===

# Train/Test Split
random.shuffle(data)
split_idx = int(0.9 * len(data))
train_data = data[:split_idx]
test_data = data[split_idx:]

train_loader = DataLoader(train_data, batch_size=8, shuffle=True)
test_loader = DataLoader(test_data, batch_size=8)

def train():
    model.train()
    total_loss = 0
    for batch in train_loader:
        optimizer.zero_grad()
        out = model(batch.x, batch.internal_node_data, batch.level, batch.edge_index)
        loss = criterion(out, batch.y.float())
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(train_loader)

@torch.no_grad()
def evaluate(loader, threshold=0.5):
    model.eval()
    total_correct = 0
    total_nodes = 0
    tp, fp, fn = 0, 0, 0

    for batch in loader:
        out = model(batch.x, batch.internal_node_data, batch.level, batch.edge_index)
        preds = torch.sigmoid(out) > threshold
        total_correct += (preds == batch.y.bool()).sum().item()
        total_nodes += batch.y.size(0)

        tp += ((preds == 1) & (batch.y == 1)).sum().item()
        fp += ((preds == 1) & (batch.y == 0)).sum().item()
        fn += ((preds == 0) & (batch.y == 1)).sum().item()

    acc = total_correct / total_nodes
    precision = tp / (tp + fp + 1e-8)
    recall = tp / (tp + fn + 1e-8)
    f1 = 2 * precision * recall / (precision + recall + 1e-8)
    return acc, precision, recall, f1

@torch.no_grad()
def find_best_threshold(loader, thresholds=np.linspace(0, 1, 101)):
    model.eval()
    best_threshold = 0.5
    best_f1 = 0.0

    # Alle Outputs und Labels sammeln, damit man nicht für jeden Threshold neu durch die Daten geht
    all_outs = []
    all_labels = []
    for batch in loader:
        out = model(batch.x, batch.internal_node_data, batch.level, batch.edge_index)
        all_outs.append(torch.sigmoid(out))
        all_labels.append(batch.y)
    all_outs = torch.cat(all_outs)
    all_labels = torch.cat(all_labels)

    for threshold in thresholds:
        preds = all_outs > threshold
        tp = ((preds == 1) & (all_labels == 1)).sum().item()
        fp = ((preds == 1) & (all_labels == 0)).sum().item()
        fn = ((preds == 0) & (all_labels == 1)).sum().item()

        precision = tp / (tp + fp + 1e-8)
        recall = tp / (tp + fn + 1e-8)
        f1 = 2 * precision * recall / (precision + recall + 1e-8)

        if f1 > best_f1:
            best_f1 = f1
            best_threshold = threshold

    return best_threshold, best_f1

# === 3. Training starten ===
for epoch in range(1, 21):
    loss = train()
    acc, prec, rec, f1 = evaluate(train_loader)
    print(f"Train: Epoch {epoch:02d} | Loss: {loss:.4f} | Acc: {acc:.3f} | Prec: {prec:.3f} | Rec: {rec:.3f} | F1: {f1:.3f}")
    acc, prec, rec, f1 = evaluate(test_loader)
    print(f"Test:  Epoch {epoch:02d} | Loss: {loss:.4f} | Acc: {acc:.3f} | Prec: {prec:.3f} | Rec: {rec:.3f} | F1: {f1:.3f}")

model_save = model

# Nach Training besten Threshold bestimmen
best_threshold, best_f1 = find_best_threshold(train_loader)
print(f"\nBester Threshold Train: {best_threshold:.3f} mit F1-Score: {best_f1:.3f}")

# Evaluation mit bestem Threshold
best_threshold, best_f1 = find_best_threshold(test_loader)
print(f"\nBester Threshold Test:  {best_threshold:.3f} mit F1-Score: {best_f1:.3f}")
acc, prec, rec, f1 = evaluate(test_loader, threshold=best_threshold)
print(f"Evaluation mit bestem Threshold:")
print(f"Acc: {acc:.3f} | Prec: {prec:.3f} | Rec: {rec:.3f} | F1: {f1:.3f}")

/tmp/ipykernel_25057/2686088309.py:15: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  pos_weight = torch.tensor((ratio**0.3), dtype=torch.float)


Pos Weight: 3.53
Train: Epoch 01 | Loss: 0.1194 | Acc: 0.982 | Prec: 0.400 | Rec: 0.447 | F1: 0.422
Test:  Epoch 01 | Loss: 0.1194 | Acc: 0.982 | Prec: 0.397 | Rec: 0.436 | F1: 0.416
Train: Epoch 02 | Loss: 0.0737 | Acc: 0.981 | Prec: 0.430 | Rec: 0.813 | F1: 0.563
Test:  Epoch 02 | Loss: 0.0737 | Acc: 0.981 | Prec: 0.426 | Rec: 0.791 | F1: 0.554
Train: Epoch 03 | Loss: 0.0556 | Acc: 0.986 | Prec: 0.526 | Rec: 0.812 | F1: 0.639
Test:  Epoch 03 | Loss: 0.0556 | Acc: 0.986 | Prec: 0.524 | Rec: 0.791 | F1: 0.630
Train: Epoch 04 | Loss: 0.0502 | Acc: 0.983 | Prec: 0.462 | Rec: 0.905 | F1: 0.611
Test:  Epoch 04 | Loss: 0.0502 | Acc: 0.983 | Prec: 0.463 | Rec: 0.887 | F1: 0.608
Train: Epoch 05 | Loss: 0.0472 | Acc: 0.985 | Prec: 0.499 | Rec: 0.895 | F1: 0.641
Test:  Epoch 05 | Loss: 0.0472 | Acc: 0.985 | Prec: 0.496 | Rec: 0.875 | F1: 0.633
Train: Epoch 06 | Loss: 0.0456 | Acc: 0.986 | Prec: 0.506 | Rec: 0.900 | F1: 0.648
Test:  Epoch 06 | Loss: 0.0456 | Acc: 0.985 | Prec: 0.502 | Rec: 0.866

In [27]:
import NN_visualization
reload(NN_visualization)

model_save = model

NN_visualization.save_RecipientFinder(model, best_threshold=best_threshold)



Model saved to /mnt/c/Users/uhewm/Desktop/ProjectHGT/recipient_finder_model.pt


In [69]:
# Nach Training besten Threshold bestimmen
best_threshold, best_f1 = find_best_threshold(train_loader)
print(f"\nBester Threshold Train: {best_threshold:.3f} mit F1-Score: {best_f1:.3f}")

# Evaluation mit bestem Threshold
best_threshold, best_f1 = find_best_threshold(test_loader)
print(f"\nBester Threshold Test:  {best_threshold:.3f} mit F1-Score: {best_f1:.3f}")
acc, prec, rec, f1 = evaluate(test_loader, threshold=best_threshold)
print(f"Evaluation mit bestem Threshold:")
print(f"Acc: {acc:.3f} | Prec: {prec:.3f} | Rec: {rec:.3f} | F1: {f1:.3f}")


Bester Threshold Train: 0.670 mit F1-Score: 0.728

Bester Threshold Test:  0.670 mit F1-Score: 0.710
Evaluation mit bestem Threshold:
Acc: 0.993 | Prec: 0.804 | Rec: 0.636 | F1: 0.710


In [61]:
# Nach Training besten Threshold bestimmen
best_threshold, best_f1 = find_best_threshold(train_loader)
print(f"\nBester Threshold Train: {best_threshold:.3f} mit F1-Score: {best_f1:.3f}")

# Evaluation mit bestem Threshold
best_threshold, best_f1 = find_best_threshold(test_loader)
print(f"\nBester Threshold Test:  {best_threshold:.3f} mit F1-Score: {best_f1:.3f}")
acc, prec, rec, f1 = evaluate(test_loader, threshold=best_threshold)
print(f"Evaluation mit bestem Threshold:")
print(f"Acc: {acc:.3f} | Prec: {prec:.3f} | Rec: {rec:.3f} | F1: {f1:.3f}")


Bester Threshold Train: 0.700 mit F1-Score: 0.851

Bester Threshold Test:  0.680 mit F1-Score: 0.758
Evaluation mit bestem Threshold:
Acc: 0.994 | Prec: 0.760 | Rec: 0.756 | F1: 0.758


In [18]:
import torch
from pyvis.network import Network
import subprocess
import webbrowser
from pathlib import Path
import networkx as nx

reload(Data_Loader)

random_file = random.choice(files)

d = Data_Loader.load_file(random_file)
single_graph = Data_Loader.aggregate_sequences(d)

single_data = Data_Loader.graph_to_dataset_RecipientFinder(single_graph)

"""
for node, attrs in single_graph.nodes(data=True):
    print(node, attrs)

print(single_data.x)
"""

hgt_nodes = [node for node in single_graph.nodes() if single_graph.nodes[node]["recipient"]["is_parent_node"] ]
gene_absence_presence_matrix = [single_graph.nodes[node]["gene_present_below_node"] > 0 for node in single_graph.nodes() if node < 100]

# === 1. Modellvorhersagen berechnen ===
model.eval()
with torch.no_grad():
    logits = model(single_data.x, single_data.internal_node_data, single_data.level, single_data.edge_index)  # Shape: [num_nodes]
    probs = torch.sigmoid(logits).cpu().numpy()  # Werte zwischen 0 und 1

# Map von Node-ID zu Wahrscheinlichkeit
pred_probs = {i: p > best_threshold for i, p in enumerate(probs)}
pred_nodes = sorted([i for i, flag in pred_probs.items() if flag])

node_to_id = {node: i for i, node in enumerate(single_graph.nodes)}
print("Predicted Nodes: ", sorted([node_to_id[i] for i, flag in pred_probs.items() if flag]))
print("True Nodes: ", sorted(set(hgt_nodes)))

pred_probs = {node_to_id[i]: p for i, p in enumerate(probs)}


# --- x/y Koordinaten für Blätter und innere Knoten berechnen ---
x_spacing = 100
y_spacing = 100

node_x = {}
node_y = {}

# Maximaler Level aus single_graph
max_level = max(single_graph.nodes[n].get("level", 0) for n in single_graph.nodes)

# Hilfsfunktion: finde alle Blätter unterhalb eines Knotens
def get_descendant_leaves(G, node):
    """Alle Blätter, die von `node` erreichbar sind (rekursiv)."""
    stack = list(G.successors(node))
    reachable_leaves = []
    while stack:
        temp_node = stack.pop()
        children = list(G.successors(temp_node))
        if len(children) > 0:
            stack.extend(children)
        else:
            reachable_leaves.append(temp_node)
    return reachable_leaves

# === Blätter (Level 0) oben ===
leaves = [n for n in single_graph.nodes if single_graph.nodes[n].get("level", 0) == 0]
for i, node in enumerate(sorted(leaves)):  
    node_x[node] = i * x_spacing
    node_y[node] = (max_level - 0) * y_spacing  # Blätter oben

# === Innere Knoten: levelweise platzieren ===
levels_in_graph = sorted(set(nx.get_node_attributes(single_graph, "level").values()))
for level in levels_in_graph[1:]:  # 0 schon behandelt
    nodes_in_level = [n for n in single_graph.nodes if single_graph.nodes[n].get("level", 0) == level]
    for node in nodes_in_level:
        # Finde alle Blätter unterhalb
        reachable_leaves = get_descendant_leaves(single_graph, node)
        if reachable_leaves:
            leaf_x = [node_x[l] for l in reachable_leaves if l in node_x]
            node_x[node] = np.mean(leaf_x)
        else:
            node_x[node] = 0
        node_y[node] = (max_level - level) * y_spacing
   
# === Netzwerk initialisieren (Hierarchical Layout deaktiviert!) ===
net = Network(height="900px", width="100%", directed=True)

net.set_options("""
{
  "nodes": {
    "shape": "dot",
    "size": 12,
    "font": { "size": 30 }
  },
  "edges": {
    "arrows": {
      "to": { "enabled": true, "scaleFactor": 0.5 }
    }
  },
  "physics": {
    "enabled": false
  }
}
""")
# === Knoten hinzufügen mit festen x/y ===

ATTRS = [
    "sum_seq",
    "tree_length",
    "time",
    "pred"
]

for node in single_graph.nodes():
    values = {}

    for key in ATTRS:
        if key == "pred":
            values[key] = float(pred_probs[node])
        else:
            values[key] = float(single_graph.nodes[node].get(key, 0))

    # Title-String
    title = ", ".join([f"{k}: {values[k]:.2f}" for k in ATTRS])

    # Label-String (mit Zeilenumbrüchen)
    label_values = ", ".join([f"{values[k]:.2f}" for k in ATTRS])
    label = f"{node}\n({label_values})"

    # Farbe
    if node in hgt_nodes and pred_probs[node] > best_threshold:
        color = "green"
    elif node not in hgt_nodes and pred_probs[node] > best_threshold:
        color = "violet"
    elif node in hgt_nodes and pred_probs[node] <= best_threshold:
        color = "red"
    elif node < 100 and gene_absence_presence_matrix[node] == 1:
        color = "orange"
    elif node < 100 and gene_absence_presence_matrix[node] == 0:
        color = "black"
    else:
        color = "lightblue"


    net.add_node(node, label=label, title=title, color=color,
                 x=node_x[node], y=node_y[node])

# === Kanten hinzufügen ===
for u, v in single_graph.edges():
    net.add_edge(u, v)

# === HTML-Datei speichern und direkt in Chrome öffnen ===
html_file = Path("/mnt/c/Users/uhewm/OneDrive/PhD/Project No.2/pangenome/graph.html")
html_file = Path("/mnt/c/ProjectHGT/graph.html")
html_file.parent.mkdir(parents=True, exist_ok=True)
net.show(str(html_file), notebook=False)

# WSL-Pfad in Windows-Pfad umwandeln
win_path = subprocess.run(["wslpath", "-w", str(html_file)], capture_output=True, text=True).stdout.strip()

# Direkt in Chrome öffnen
subprocess.run(["cmd.exe", "/C", "start", "chrome", win_path])


/mnt/c/Users/uhewm/OneDrive/PhD/Project No.2/pangenome-gene-transfer-simulation/Data_Loader.py:96: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  edges = torch.tensor(graph_properties[1], dtype=torch.long)  # [2, num_edges]
/mnt/c/Users/uhewm/OneDrive/PhD/Project No.2/pangenome-gene-transfer-simulation/Data_Loader.py:97: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  coords = torch.tensor(graph_properties[2].T)             # [2, num_nodes]


Predicted Nodes:  [188, 190, 196, 197, 198]
True Nodes:  [190, 196, 197]
/mnt/c/ProjectHGT/graph.html
>4;1H84l=                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                      

CompletedProcess(args=['cmd.exe', '/C', 'start', 'chrome', 'C:\\ProjectHGT\\graph.html'], returncode=0)